In [4]:
import sys

print("Python version:", sys.version)
print("Python executable:", sys.executable)

Python version: 3.14.4 (tags/v3.14.4:23116f9, Apr  7 2026, 14:10:54) [MSC v.1944 64 bit (AMD64)]
Python executable: c:\Users\Gulso\Documents\Gul\CredBlock-FL\.venv\Scripts\python.exe


In [5]:
import pandas as pd

DATA_PATH = "../data/rba-dataset.csv"

# Read only the first 100,000 login records
df = pd.read_csv(DATA_PATH, nrows=100_000)

print("Dataset shape:", df.shape)

df.head()

Dataset shape: (100000, 16)


,index,Login Timestamp,User ID,Round-Trip Time [ms],IP Address,Country,Region,City,ASN,User Agent String,Browser Name and Version,OS Name and Version,Device Type,Login Successful,Is Attack IP,Is Account Takeover
0,0,2020-02-03 12:43:30.772,-4324475583306591935,NaN,10.0.65.171,NO,-,-,29695,Mozilla/5.0 (iPhone; CPU iPhone OS 13_4 like ...,Firefox 20.0.0.1618,iOS 13.4,mobile,False,False,False
1,1,2020-02-03 12:43:43.549,-4324475583306591935,NaN,194.87.207.6,AU,-,-,60117,Mozilla/5.0 (Linux; Android 4.1; Galaxy Nexus...,Chrome Mobile 46.0.2490,Android 4.1,mobile,False,False,False
2,2,2020-02-03 12:43:55.873,-3284137479262433373,NaN,81.167.144.58,NO,Vestland,Urangsvag,29695,Mozilla/5.0 (iPad; CPU OS 7_1 like Mac OS X) ...,Android 2.3.3.2672,iOS 7.1,mobile,True,False,False
3,3,2020-02-03 12:43:56.180,-4324475583306591935,NaN,170.39.78.152,US,-,-,393398,Mozilla/5.0 (Linux; Android 4.1; Galaxy Nexus...,Chrome Mobile WebView 85.0.4183,Android 4.1,mobile,False,False,False
4,4,2020-02-03 12:43:59.396,-4618854071942621186,NaN,10.0.0.47,US,Virginia,Ashburn,398986,Mozilla/5.0 (Linux; U; Android 2.2) Build/NMA...,Chrome Mobile WebView 85.0.4183,Android 2.2,mobile,False,True,False


In [6]:
# Basic information about the 100,000-row sample

print("Total login records:", len(df))
print("Unique users:", df["User ID"].nunique())
print("Unique IP addresses:", df["IP Address"].nunique())

print("\nMissing values:")
print(df.isnull().sum())

print("\nLogin Successful:")
print(df["Login Successful"].value_counts())

print("\nAttack IP:")
print(df["Is Attack IP"].value_counts())

print("\nAccount Takeover:")
print(df["Is Account Takeover"].value_counts())

Total login records: 100000
Unique users: 41500
Unique IP addresses: 45437

Missing values:
index                           0
Login Timestamp                 0
User ID                         0
Round-Trip Time [ms]        95718
IP Address                      0
Country                         0
Region                         78
City                           35
ASN                             0
User Agent String               0
Browser Name and Version        0
OS Name and Version             0
Device Type                     7
Login Successful                0
Is Attack IP                    0
Is Account Takeover             0
dtype: int64

Login Successful:
Login Successful
False    51273
True     48727
Name: count, dtype: int64

Attack IP:
Is Attack IP
False    90302
True      9698
Name: count, dtype: int64

Account Takeover:
Is Account Takeover
False    99998
True         2
Name: count, dtype: int64


In [7]:
def show_distribution(column):
    counts = df[column].value_counts()
    percentages = df[column].value_counts(normalize=True) * 100

    result = pd.DataFrame({
        "Count": counts,
        "Percentage (%)": percentages.round(3)
    })

    print(f"\n{column}")
    print(result)


show_distribution("Login Successful")
show_distribution("Is Attack IP")
show_distribution("Is Account Takeover")


Login Successful
                  Count  Percentage (%)
Login Successful                       
False             51273          51.273
True              48727          48.727

Is Attack IP
              Count  Percentage (%)
Is Attack IP                       
False         90302          90.302
True           9698           9.698

Is Account Takeover
                     Count  Percentage (%)
Is Account Takeover                       
False                99998          99.998
True                     2           0.002


In [8]:
with open("../data/README.md", "r", encoding="utf-8") as file:
    readme = file.read()

print(readme)

# Login Data Set for Risk-Based Authentication

> Synthesized login feature data of >33M login attempts and >3.3M users
> on a large-scale online service in Norway. Original data collected
> between February 2020 and February 2021.

This data sets aims to foster research and development for
[Risk-Based Authentication (RBA)] systems. The data was synthesized from
the real-world login behavior of more than 3.3M users at a large-scale
single sign-on (SSO) online service in Norway.

The users used this SSO to access sensitive data provided by the online
service, e.g., a cloud storage and billing information. We used this
data set to study how the [Freeman et al. (2016)] RBA model behaves on a
large-scale online service in the real world (see [Publication](#publication)). The
synthesized data set can reproduce these results made on the original
data set (see [Study Reproduction](#study-reproduction)). Beyond that, you can use this data
set to evaluate and improve RBA algorithms under real-w

In [9]:
terms = [
    "Login Successful",
    "Is Attack IP",
    "Is Account Takeover"
]

lines = readme.splitlines()

for term in terms:
    print(f"\n===== {term} =====")

    for i, line in enumerate(lines):
        if term.lower() in line.lower():
            start = max(0, i - 3)
            end = min(len(lines), i + 6)

            for nearby_line in lines[start:end]:
                print(nearby_line)


===== Login Successful =====
User ID                    | Integer   | Idenfication number related to the affected user account                                         | [Random pseudonym]
Login Timestamp            | Integer   | Timestamp related to the login attempt                                                           | [64 Bit timestamp]
Round-Trip Time (RTT) [ms] | Integer   | Server-side measured latency between client and server                                           | 1 - 8600000
Login Successful           | Boolean   | `True`: Login was successful, `False`: Login failed                                              | (`true`, `false`)
Is Attack IP               | Boolean   | IP address was found in known attacker data set                                                  | (`true`, `false`)
Is Account Takeover        | Boolean   | Login attempt was identified as account takeover by incident response team of the online service | (`true`, `false`)

[^1]: Few (invalid) user 

In [10]:
# Process the full RBA dataset in manageable chunks

CHUNK_SIZE = 500_000

total_rows = 0
login_success_counts = {}
attack_ip_counts = {}
takeover_counts = {}

reader = pd.read_csv(
    DATA_PATH,
    chunksize=CHUNK_SIZE,
    usecols=[
        "Login Successful",
        "Is Attack IP",
        "Is Account Takeover"
    ]
)

for chunk_number, chunk in enumerate(reader, start=1):
    total_rows += len(chunk)

    # Count each True/False label in this chunk
    for column, storage in [
        ("Login Successful", login_success_counts),
        ("Is Attack IP", attack_ip_counts),
        ("Is Account Takeover", takeover_counts)
    ]:
        counts = chunk[column].value_counts()

        for value, count in counts.items():
            storage[value] = storage.get(value, 0) + count

    print(f"Processed chunk {chunk_number}: {total_rows:,} rows")

print("\n--- FULL DATASET RESULTS ---")
print("Total rows:", f"{total_rows:,}")
print("Login Successful:", login_success_counts)
print("Is Attack IP:", attack_ip_counts)
print("Is Account Takeover:", takeover_counts)

Processed chunk 1: 500,000 rows
Processed chunk 2: 1,000,000 rows
Processed chunk 3: 1,500,000 rows
Processed chunk 4: 2,000,000 rows
Processed chunk 5: 2,500,000 rows
Processed chunk 6: 3,000,000 rows
Processed chunk 7: 3,500,000 rows
Processed chunk 8: 4,000,000 rows
Processed chunk 9: 4,500,000 rows
Processed chunk 10: 5,000,000 rows
Processed chunk 11: 5,500,000 rows
Processed chunk 12: 6,000,000 rows
Processed chunk 13: 6,500,000 rows
Processed chunk 14: 7,000,000 rows
Processed chunk 15: 7,500,000 rows
Processed chunk 16: 8,000,000 rows
Processed chunk 17: 8,500,000 rows
Processed chunk 18: 9,000,000 rows
Processed chunk 19: 9,500,000 rows
Processed chunk 20: 10,000,000 rows
Processed chunk 21: 10,500,000 rows
Processed chunk 22: 11,000,000 rows
Processed chunk 23: 11,500,000 rows
Processed chunk 24: 12,000,000 rows
Processed chunk 25: 12,500,000 rows
Processed chunk 26: 13,000,000 rows
Processed chunk 27: 13,500,000 rows
Processed chunk 28: 14,000,000 rows
Processed chunk 29: 14

In [11]:
# Count missing values across the complete RBA dataset

CHUNK_SIZE = 500_000

missing_counts = None
total_rows = 0

reader = pd.read_csv(
    DATA_PATH,
    chunksize=CHUNK_SIZE
)

for chunk_number, chunk in enumerate(reader, start=1):
    
    chunk_missing = chunk.isnull().sum()

    if missing_counts is None:
        missing_counts = chunk_missing
    else:
        missing_counts = missing_counts.add(chunk_missing, fill_value=0)

    total_rows += len(chunk)

    print(f"Processed chunk {chunk_number}: {total_rows:,} rows")


missing_percentages = (missing_counts / total_rows * 100).round(3)

missing_results = pd.DataFrame({
    "Missing Count": missing_counts.astype(int),
    "Missing Percentage (%)": missing_percentages
})

print("\n--- FULL DATASET MISSING VALUES ---")
print(missing_results)

Processed chunk 1: 500,000 rows
Processed chunk 2: 1,000,000 rows
Processed chunk 3: 1,500,000 rows
Processed chunk 4: 2,000,000 rows
Processed chunk 5: 2,500,000 rows
Processed chunk 6: 3,000,000 rows
Processed chunk 7: 3,500,000 rows
Processed chunk 8: 4,000,000 rows
Processed chunk 9: 4,500,000 rows
Processed chunk 10: 5,000,000 rows
Processed chunk 11: 5,500,000 rows
Processed chunk 12: 6,000,000 rows
Processed chunk 13: 6,500,000 rows
Processed chunk 14: 7,000,000 rows
Processed chunk 15: 7,500,000 rows
Processed chunk 16: 8,000,000 rows
Processed chunk 17: 8,500,000 rows
Processed chunk 18: 9,000,000 rows
Processed chunk 19: 9,500,000 rows
Processed chunk 20: 10,000,000 rows
Processed chunk 21: 10,500,000 rows
Processed chunk 22: 11,000,000 rows
Processed chunk 23: 11,500,000 rows
Processed chunk 24: 12,000,000 rows
Processed chunk 25: 12,500,000 rows
Processed chunk 26: 13,000,000 rows
Processed chunk 27: 13,500,000 rows
Processed chunk 28: 14,000,000 rows
Processed chunk 29: 14

In [12]:
# Count unique users and IP addresses across the full dataset

CHUNK_SIZE = 500_000

unique_users = set()
unique_ips = set()
total_rows = 0

reader = pd.read_csv(
    DATA_PATH,
    chunksize=CHUNK_SIZE,
    usecols=["User ID", "IP Address"]
)

for chunk_number, chunk in enumerate(reader, start=1):

    unique_users.update(chunk["User ID"].unique())
    unique_ips.update(chunk["IP Address"].unique())

    total_rows += len(chunk)

    print(f"Processed chunk {chunk_number}: {total_rows:,} rows")

print("\n--- FULL DATASET GRAPH ENTITIES ---")
print("Total login records:", f"{total_rows:,}")
print("Unique users:", f"{len(unique_users):,}")
print("Unique IP addresses:", f"{len(unique_ips):,}")
print("Potential User + IP nodes:", f"{len(unique_users) + len(unique_ips):,}")

Processed chunk 1: 500,000 rows
Processed chunk 2: 1,000,000 rows
Processed chunk 3: 1,500,000 rows
Processed chunk 4: 2,000,000 rows
Processed chunk 5: 2,500,000 rows
Processed chunk 6: 3,000,000 rows
Processed chunk 7: 3,500,000 rows
Processed chunk 8: 4,000,000 rows
Processed chunk 9: 4,500,000 rows
Processed chunk 10: 5,000,000 rows
Processed chunk 11: 5,500,000 rows
Processed chunk 12: 6,000,000 rows
Processed chunk 13: 6,500,000 rows
Processed chunk 14: 7,000,000 rows
Processed chunk 15: 7,500,000 rows
Processed chunk 16: 8,000,000 rows
Processed chunk 17: 8,500,000 rows
Processed chunk 18: 9,000,000 rows
Processed chunk 19: 9,500,000 rows
Processed chunk 20: 10,000,000 rows
Processed chunk 21: 10,500,000 rows
Processed chunk 22: 11,000,000 rows
Processed chunk 23: 11,500,000 rows
Processed chunk 24: 12,000,000 rows
Processed chunk 25: 12,500,000 rows
Processed chunk 26: 13,000,000 rows
Processed chunk 27: 13,500,000 rows
Processed chunk 28: 14,000,000 rows
Processed chunk 29: 14

In [13]:
# Count unique User-IP pairs across the full dataset

CHUNK_SIZE = 500_000

unique_pairs = set()
total_rows = 0

reader = pd.read_csv(
    DATA_PATH,
    chunksize=CHUNK_SIZE,
    usecols=["User ID", "IP Address"]
)

for chunk_number, chunk in enumerate(reader, start=1):

    pairs = zip(chunk["User ID"], chunk["IP Address"])
    unique_pairs.update(pairs)

    total_rows += len(chunk)

    print(f"Processed chunk {chunk_number}: {total_rows:,} rows")

print("\n--- USER-IP RELATIONSHIP RESULTS ---")
print("Total login attempts:", f"{total_rows:,}")
print("Unique User-IP pairs:", f"{len(unique_pairs):,}")
print(
    "Repeated User-IP interactions:",
    f"{total_rows - len(unique_pairs):,}"
)
print(
    "Unique-pair ratio:",
    f"{(len(unique_pairs) / total_rows) * 100:.3f}%"
)

Processed chunk 1: 500,000 rows
Processed chunk 2: 1,000,000 rows
Processed chunk 3: 1,500,000 rows
Processed chunk 4: 2,000,000 rows
Processed chunk 5: 2,500,000 rows
Processed chunk 6: 3,000,000 rows
Processed chunk 7: 3,500,000 rows
Processed chunk 8: 4,000,000 rows
Processed chunk 9: 4,500,000 rows
Processed chunk 10: 5,000,000 rows
Processed chunk 11: 5,500,000 rows
Processed chunk 12: 6,000,000 rows
Processed chunk 13: 6,500,000 rows
Processed chunk 14: 7,000,000 rows
Processed chunk 15: 7,500,000 rows
Processed chunk 16: 8,000,000 rows
Processed chunk 17: 8,500,000 rows
Processed chunk 18: 9,000,000 rows
Processed chunk 19: 9,500,000 rows
Processed chunk 20: 10,000,000 rows
Processed chunk 21: 10,500,000 rows
Processed chunk 22: 11,000,000 rows
Processed chunk 23: 11,500,000 rows
Processed chunk 24: 12,000,000 rows
Processed chunk 25: 12,500,000 rows
Processed chunk 26: 13,000,000 rows
Processed chunk 27: 13,500,000 rows
Processed chunk 28: 14,000,000 rows
Processed chunk 29: 14